# FreeSpace в контейнере

Анализатор дискового пространства: показывает, чем занято место, ищет папки по
имени (все `venv` разом), удаляет в корзину с возможностью вернуть.

**Что делать:** впишите токен SberOSC в ячейку «Шаг 1» и выполните обе
ячейки по порядку. Вторая поднимет сервер, соберёт адрес за
`jupyter-server-proxy` и покажет приложение прямо здесь, в `iframe`, а заодно
попробует открыть его в новой вкладке.

Если вкладка не открылась — её заблокировал браузер; пользуйтесь кнопкой
«Открыть FreeSpace» или работайте прямо в `iframe`.


## Шаг 1. Токен SberOSC

`fastapi` и `uvicorn` ставятся не с PyPI — из контейнера туда хода нет, — а из
индекса пакетов портала, и он требует токен. Возьмите токен в SberOSC, впишите
его в ячейку ниже и выполните её.

Токен нужен только для установки. Если зависимости уже стоят, оставьте строку
пустой: ячейка запуска ничего ставить не станет.

In [ ]:
# Токен SberOSC. Без него pip не достучится до индекса пакетов портала.
#
# Значение остаётся только в памяти ядра: в напечатанных командах оно
# заменяется звёздочками, потому что вывод ячеек сохраняется в самом .ipynb и
# уезжает в репозиторий вместе с ним.

TOKEN = ""

## Шаг 2. Запуск

In [ ]:
# FreeSpace — запуск в контейнере с Jupyter.
#
# Выполните эту ячейку. Она поднимет бэкенд, соберёт адрес за
# jupyter-server-proxy и откроет страницу. Повторный запуск ячейки гасит
# предыдущий сервер, а не плодит новые.

import json
import os
import socket
import subprocess
import sys
import time
import urllib.error
import urllib.request
from pathlib import Path

from IPython.display import HTML, Javascript, display

# Внешний адрес контейнера: именно на него смотрит браузер, тогда как сервер
# слушает петлевой адрес внутри пода.
DOMAIN = "https://jupyterhub-datalab.apps.prom-datalab.ca.sbrf.ru"


def find_project_root(start: Path) -> Path:
    """Каталог, в котором лежит пакет freespace: тетрадку могли открыть откуда угодно."""
    for candidate in [start, *start.parents]:
        if (candidate / "freespace" / "web" / "api.py").exists():
            return candidate
    return start


PROJECT = find_project_root(Path.cwd().resolve())
STATE = PROJECT / ".freespace-server.json"
LOG = PROJECT / ".freespace-server.log"
os.chdir(PROJECT)


def stop_previous() -> None:
    """Погасить сервер, поднятый прошлым запуском этой же ячейки."""
    if not STATE.exists():
        return
    try:
        pid = json.loads(STATE.read_text())["pid"]
    except (OSError, ValueError, KeyError):
        STATE.unlink(missing_ok=True)
        return
    try:
        os.kill(pid, 15)
        for _ in range(30):
            time.sleep(0.1)
            os.kill(pid, 0)          # бросит OSError, когда процесс исчезнет
        os.kill(pid, 9)              # не отреагировал на TERM — добиваем
    except OSError:
        pass
    STATE.unlink(missing_ok=True)
    print(f"Остановлен прежний сервер (pid {pid}).")


# Индекс пакетов портала. Обычный `pip install` из контейнера не работает:
# наружу, на PyPI, хода нет, и ставить можно только отсюда — по токену.
INDEX_HOST = "sberosc.ca.sbrf.ru"
INDEX_PATH = "/repo/pypi/simple"


def install(packages: list[str], token: str) -> None:
    """Поставить пакеты из индекса портала.

    Вывод pip не глушится: установка идёт минуту-другую, и молчащая ячейка в
    это время неотличима от зависшей. Команда печатается целиком — если
    установка не удалась, её можно повторить руками в терминале.
    """
    command = [sys.executable, "-m", "pip", "install", "--user",
               "--disable-pip-version-check"]
    if token:
        command += [
            f"--index-url=https://token:{token}@{INDEX_HOST}{INDEX_PATH}",
            f"--trusted-host={INDEX_HOST}",
        ]
    command += packages

    # Токен в напечатанной команде затирается: вывод ячейки сохраняется в самом
    # .ipynb и уезжает в репозиторий вместе с ним.
    printable = " ".join(command).replace(token, "***") if token else " ".join(command)
    print(f"Ставлю зависимости портала: {', '.join(packages)}")
    print(f"$ {printable}")
    result = subprocess.run(command)
    if result.returncode != 0:
        raise RuntimeError(
            "не удалось установить зависимости портала. Проверьте токен SberOSC и "
            "доступность индекса; полная команда напечатана выше — её можно "
            "выполнить вручную в терминале."
        )


def ensure_deps(token: str) -> str:
    """Поставить fastapi и uvicorn, если их нет."""
    try:
        import fastapi  # noqa: F401
        import uvicorn  # noqa: F401
        return "уже стоят"
    except ImportError:
        pass
    if not token:
        raise RuntimeError(
            "fastapi и uvicorn не установлены, а TOKEN пуст. Впишите токен SberOSC "
            "в ячейку «Шаг 1» и выполните её: без токена индекс пакетов портала "
            "недоступен, а с PyPI из контейнера связи нет."
        )
    install(["fastapi>=0.110", "uvicorn>=0.27"], token)
    return "поставлены из индекса портала"


def free_port(start: int = 8000, tries: int = 50) -> int:
    for port in range(start, start + tries):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            try:
                sock.bind(("127.0.0.1", port))
                return port
            except OSError:
                continue
    raise OSError(f"нет свободного порта в диапазоне {start}..{start + tries - 1}")


def proxy_prefix() -> str:
    """Префикс URL, который jupyter-server-proxy ожидает увидеть."""
    prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX") or os.environ.get("NB_PREFIX")
    if prefix:
        return prefix if prefix.endswith("/") else prefix + "/"
    user = os.environ.get("JUPYTERHUB_USER")
    return f"/user/{user}/" if user else ""


stop_previous()
# Ячейку запуска нередко выполняют первой, не тронув ячейку с токеном. Пустой
# токен — не беда, если зависимости уже стоят; ensure_deps скажет, если нет.
print("Зависимости:", ensure_deps(globals().get("TOKEN", "")))

try:
    import jupyter_server_proxy  # noqa: F401
    PROXY_OK = True
except ImportError:
    PROXY_OK = False

PORT = free_port()
PREFIX = proxy_prefix()
# За прокси приложение живёт по адресу <префикс>proxy/<порт>/. Тот же префикс
# уходит серверу как root_path, иначе FastAPI сгенерирует ссылки от корня.
ROOT_PATH = f"{PREFIX}proxy/{PORT}" if PREFIX else ""
URL = f"{DOMAIN}{ROOT_PATH}/" if PREFIX else f"http://127.0.0.1:{PORT}/"

command = [sys.executable, "-m", "freespace.web", "--host", "127.0.0.1", "--port", str(PORT)]
if ROOT_PATH:
    command += ["--root-path", ROOT_PATH]

with open(LOG, "wb") as log:
    server = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, cwd=PROJECT)
STATE.write_text(json.dumps({"pid": server.pid, "port": PORT, "url": URL}))

# Ждём, пока порт начнёт отвечать: сразу открывать страницу нельзя, иначе
# браузер увидит «connection refused» и пользователь решит, что не работает.
ready = False
for _ in range(150):
    if server.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/config", timeout=1)
        ready = True
        break
    except (urllib.error.URLError, OSError):
        time.sleep(0.2)

if not ready:
    print("Сервер не поднялся. Последние строки лога:\n")
    print(LOG.read_text(errors="replace")[-3000:])
else:
    print(f"Сервер работает: pid {server.pid}, порт {PORT}.")
    if PREFIX and not PROXY_OK:
        print(
            "\n!! В окружении не найден jupyter_server_proxy — ссылка через прокси\n"
            "   работать не будет. Поставьте его (pip install jupyter-server-proxy)\n"
            "   и перезапустите Jupyter, либо пользуйтесь диагностикой из spike.py."
        )
    display(HTML(f"""
      <div style="font:14px/1.5 system-ui,sans-serif;padding:10px 0">
        <a href="{URL}" target="_blank"
           style="display:inline-block;padding:9px 18px;border-radius:8px;
                  background:#4F86C6;color:#fff;text-decoration:none;font-weight:600">
          Открыть FreeSpace</a>
        <span style="opacity:.65;margin-left:10px">{URL}</span>
      </div>
      <iframe src="{URL}" style="width:100%;height:760px;border:1px solid #8884;
              border-radius:8px" title="FreeSpace"></iframe>
    """))
    # Открыть вкладку сама может только страница в браузере пользователя —
    # сервер тут ни при чём. Блокировщик всплывающих окон это может отменить,
    # поэтому выше уже нарисованы и ссылка, и рабочий iframe.
    display(Javascript(f"window.open({json.dumps(URL)}, '_blank');"))


## Остановить

In [ ]:
# Остановить сервер FreeSpace.

import json
import os
import time
from pathlib import Path

STATE = Path.cwd() / ".freespace-server.json"
if not STATE.exists():
    for parent in Path.cwd().parents:
        if (parent / ".freespace-server.json").exists():
            STATE = parent / ".freespace-server.json"
            break

if not STATE.exists():
    print("Запущенного сервера не найдено.")
else:
    pid = json.loads(STATE.read_text())["pid"]
    try:
        os.kill(pid, 15)
        for _ in range(30):
            time.sleep(0.1)
            os.kill(pid, 0)
        os.kill(pid, 9)
    except OSError:
        pass
    STATE.unlink(missing_ok=True)
    print(f"Сервер остановлен (pid {pid}).")


## Если что-то не так

* **Не ставятся `fastapi` и `uvicorn`.** Из контейнера нет хода на PyPI:
  зависимости берутся из индекса портала и только по токену SberOSC. Впишите
  его в ячейку «Шаг 1» и выполните её, а потом ячейку запуска. Команда `pip`
  печатается целиком (с затёртым токеном) — её можно повторить в терминале.
* **Пустая страница или 404 по ссылке.** В окружении нет `jupyter-server-proxy`.
  Ячейка запуска об этом предупреждает отдельной строкой.
* **Сервер не поднялся.** Полный лог лежит рядом с проектом в
  `.freespace-server.log`.
* **Хочется подробной диагностики окружения** — точки монтирования, свободное
  место, пригодные корни, оба варианта адреса:

  ```
  !python freespace/web/spike.py --report
  ```

* **В списке корней не то, что нужно.** По умолчанию в контейнере предлагается
  только монтирование, оканчивающееся на `/nfs`. Другой набор задаётся
  переменной `FREESPACE_ROOTS` (пути через двоеточие) до запуска сервера.
* **Удаление недоступно.** Сервер слушает петлевой адрес, поэтому удаление
  включено. Если оно выключено, значит запуск был с `--no-delete`.
